# Question Answering: point at the span, score the answer, retrieve then read

This notebook is the runnable companion to the **[Question Answering](../11-Question-Answering.md)** concept page. Every cell imports its logic from the single seeded source of truth, [`question_answering.py`](question_answering.py), so the numbers here, the prose on the page, and the figures from [`make_figures_11.py`](make_figures_11.py) cannot drift apart.

We walk the same three machines the page derives, **one idea per cell, asserting the qualitative point before printing**:

1. **The span decode** — turn answering into *pointing*: pick the best valid span `argmax_{j>=i}(S·h_i + E·h_j)` from independent start/end logits.
2. **The SQuAD scorer** — Exact Match and token-level F1 from the official definitions, and *why F1 is lenient*.
3. **Retrieve, then read** — a tiny open-domain sketch (DPR → reader, in miniature), then the live span head on a real fine-tuned model.

Everything runs **fully offline**: the live model loads only if it is already cached, otherwise a deterministic synthetic span model stands in so the notebook always completes.

## Setup — the seeded, device-honest backend

The QA math here is pure-numpy / torch-on-CPU, so the device is **honestly CPU** — the tensors are tiny and reproducibility beats throughput. The module seeds numpy once at import and uses a **stable md5 hash**, never Python's per-process-salted `hash()`, so a fresh run reproduces these numbers bit-for-bit.

In [1]:
from question_answering import runtime_banner

print(runtime_banner())

device: cpu (detected mps; pinned to CPU -- these tensors are tiny, reproducibility beats throughput)
numpy 2.4.6 | torch 2.12.0 | seed 0


## 1. The span decode — the heart of extractive QA

Extractive QA's whole trick: don't *generate* the answer, **point** at where it begins and ends. The model produces two logits per passage token — a **start** score $\mathbf{S}\cdot h_i$ and an **end** score $\mathbf{E}\cdot h_j$ — and the prediction is the valid span that maximizes their sum:

$$(\hat i, \hat j) = \operatorname*{arg\,max}_{\,j \ge i,\; j-i < L_{\max}} \big(\mathbf{S}\cdot h_i + \mathbf{E}\cdot h_j\big).$$

We use the page's hand-worked logits over the passage **[The, capital, of, France, is, Paris, .]** so the code reproduces the by-hand answer exactly.

In [2]:
from question_answering import HAND_TOKENS, HAND_START, HAND_END, best_valid_span, span_confidence

i, j, score = best_valid_span(HAND_START, HAND_END, l_max=30)
conf = span_confidence(HAND_START, HAND_END, i, j)

# the page derives the span (5,5)='Paris', score 8.5, confidence ~0.957 — assert before printing
assert (i, j) == (5, 5), f"expected span (5,5), got ({i},{j})"
assert abs(score - 8.5) < 1e-9
assert 0.95 < conf < 0.96

print(f"best valid span = ({i},{j}) -> {HAND_TOKENS[i]!r}")
print(f"span score (sum of logits) = {score:.1f}")
print(f"joint confidence P_start(i)*P_end(j) = {conf:.3f}")

best valid span = (5,5) -> 'Paris'
span score (sum of logits) = 8.5
joint confidence P_start(i)*P_end(j) = 0.957


### Why `j >= i` is not optional

The start and end heads are **independent** softmaxes — nothing stops the end head from peaking *before* the start head, which would describe a span running backwards. The decode must enumerate valid `(i, j)` pairs, not take two independent argmaxes. Let's prove it bites: hand the decoder logits whose end head peaks *before* the start head, and watch the constraint rescue a sensible span.

In [3]:
import numpy as np

# start head wants position 4; end head wants position 1 (BEFORE the start) -> backwards if naive
start = np.array([0.0, 0.0, 0.0, 0.0, 5.0, 0.0])
end = np.array([0.0, 5.0, 0.0, 0.0, 1.0, 0.0])

naive_i, naive_j = int(np.argmax(start)), int(np.argmax(end))  # the buggy two-argmax decode
valid_i, valid_j, _ = best_valid_span(start, end, l_max=30)    # the correct constrained decode

# the naive decode produces a BACKWARDS span (j < i); the constrained one never does
assert naive_j < naive_i, "this fixture is meant to trigger the backwards-span bug"
assert valid_j >= valid_i, "the constrained decode must satisfy j >= i"

print(f"naive two-argmax span: ({naive_i},{naive_j})  <- backwards! j < i describes a negative-length span")
print(f"constrained best span: ({valid_i},{valid_j})  <- valid: end never precedes start")

naive two-argmax span: (4,1)  <- backwards! j < i describes a negative-length span
constrained best span: (4,4)  <- valid: end never precedes start


### The joint span-score matrix

The decode is an argmax over a matrix: cell `(i, j)` holds `start_i + end_j`, and only the **upper triangle** `j >= i` is legal. Seeing the whole matrix makes the constraint and the winner obvious — this is the figure [`qa_joint_span_score.png`](../images/qa_joint_span_score.png) on the page, recomputed here.

In [4]:
scores = HAND_START[:, None] + HAND_END[None, :]   # scores[i, j] = start_i + end_j
n = len(HAND_TOKENS)
valid_mask = np.arange(n)[:, None] <= np.arange(n)[None, :]   # j >= i
best_flat = np.where(valid_mask, scores, -np.inf)
bi, bj = np.unravel_index(np.argmax(best_flat), scores.shape)

assert (bi, bj) == (5, 5), "argmax over the valid region must land on ('Paris','Paris')"
print(f"argmax over valid spans: ({bi},{bj}) = ({HAND_TOKENS[bi]!r},{HAND_TOKENS[bj]!r})  score={scores[bi, bj]:.1f}")
print(f"# of invalid (j<i) cells excluded: {(~valid_mask).sum()} of {n*n}")

argmax over valid spans: (5,5) = ('Paris','Paris')  score=8.5
# of invalid (j<i) cells excluded: 21 of 49


## 2. The SQuAD scorer — Exact Match and token-level F1

Both metrics first **normalize** (lowercase, drop punctuation, drop *a/an/the*, collapse whitespace). Then **EM** is all-or-nothing, while **token F1** treats each answer as a bag of tokens and gives partial credit:

$$\text{precision} = \frac{n_{\text{same}}}{|P|}, \quad \text{recall} = \frac{n_{\text{same}}}{|G|}, \quad F_1 = \frac{2\,\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}}.$$

The scorer reproduces the page's five worked pairs exactly.

In [5]:
from question_answering import EM_F1_PAIRS, exact_match, token_f1

# the two headline rows from the page — assert before printing the table
assert exact_match("Paris, France", "Paris") == 0.0 and abs(token_f1("Paris, France", "Paris") - 2/3) < 1e-9
assert exact_match("the Champ de Mars", "Champ de Mars") == 1.0  # the article normalizes away

print(f"{'prediction':22s} {'gold':18s}  EM   F1")
print("-" * 52)
for pred, gold in EM_F1_PAIRS:
    print(f"{pred!r:22s} {gold!r:18s}  {exact_match(pred, gold):.0f}   {token_f1(pred, gold):.3f}")

prediction             gold                EM   F1
----------------------------------------------------
'Paris, France'        'Paris'             0   0.667
'the Champ de Mars'    'Champ de Mars'     1   1.000
'Leonardo da Vinci'    'Leonardo da Vinci'  1   1.000
'1889'                 'in 1889'           0   0.667
'London'               'Paris'             0   0.000


### Why F1 is lenient (and EM is brutal)

Append correct-but-extra tokens to a right answer: **EM dies at the first one** (the strings no longer match exactly), but **F1 only sags** — recall stays 1.0 (you said every gold token) and precision falls slowly. This is the trade the two metrics make, and exactly the [`qa_em_f1_leniency.png`](../images/qa_em_f1_leniency.png) curve.

In [6]:
gold = "Leonardo da Vinci"
extras = ["painter", "Florentine", "Renaissance", "master", "around", "1503"]

print(f"{'k extra tokens':16s}  EM    F1")
print("-" * 32)
prev_f1 = 1.01
for k in range(len(extras) + 1):
    pred = gold + (" " + " ".join(extras[:k]) if k else "")
    em, f1 = exact_match(pred, gold), token_f1(pred, gold)
    if k == 1:
        assert em == 0.0, "one real extra token must kill EM"
    assert f1 <= prev_f1 + 1e-9, "F1 must decay monotonically as junk is appended"
    prev_f1 = f1
    print(f"{k:<16d}  {em:.0f}   {f1:.3f}")

k extra tokens    EM    F1
--------------------------------
0                 1   1.000
1                 0   0.857
2                 0   0.750
3                 0   0.667
4                 0   0.600
5                 0   0.545
6                 0   0.500


### Abstaining: the SQuAD 2.0 null score

Real questions may have **no answer in the passage**. SQuAD 2.0 reuses the `[CLS]` token (position 0) as a **null span** with score $s_{\text{null}} = \mathbf{S}\cdot h_{\text{[CLS]}} + \mathbf{E}\cdot h_{\text{[CLS]}}$; the model **abstains** when the best real span isn't enough better. Here is the decision rule firing on a synthetic example where the null score wins.

In [7]:
from question_answering import null_score

# [CLS] at position 0; positions 1..3 are passage tokens. Here nothing in the passage answers well.
start = np.array([2.0, -1.0, -1.5, -2.0])   # [CLS] start logit is the largest
end = np.array([2.0, -1.0, -1.0, -1.5])     # [CLS] end logit is the largest
passage_valid = np.array([False, True, True, True])   # spans may only land on passage tokens

_, _, s_best = best_valid_span(start, end, l_max=30, valid=passage_valid)
s_null = null_score(start, end)
tau = 0.0
decision = "answer" if s_best - s_null > tau else "abstain (no answer)"

assert s_null > s_best, "this fixture is built so the model should abstain"
assert decision.startswith("abstain")
print(f"best real span score s_best = {s_best:.2f}")
print(f"null score s_null          = {s_null:.2f}")
print(f"s_best - s_null = {s_best - s_null:.2f}  (<= tau={tau})  ->  {decision}")

best real span score s_best = -2.00
null score s_null          = 4.00
s_best - s_null = -6.00  (<= tau=0.0)  ->  abstain (no answer)


### Worked Example 3, reproduced on the real SQuAD-2.0 model

The fixture above shows the *rule*; this cell reproduces the page's **measured** Worked Example 3 by loading a SQuAD-2.0 model (`deepset/roberta-base-squad2`) and decoding the same two questions over the Apollo passage. `abstain_demo()` **asserts** each logit against the table on the page, so the numbers cannot drift. It is offline-first: if the model is not cached it falls back to the synthetic abstain fixture (the rule still fires), so this cell always completes.

In [8]:
from question_answering import abstain_demo, ABSTAIN_QUESTIONS

rows = abstain_demo()   # real roberta-base-squad2 if cached (asserts the page numbers), else synthetic fixture

# the unanswerable question must abstain; the answerable one must answer (asserted inside abstain_demo)
for (text, span_s, null_s, decision), question in zip(rows, ABSTAIN_QUESTIONS):
    print(f"span={span_s:6.2f}  null={null_s:6.2f}  -> {decision:20s}  <- {question}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

span= -6.59  null=  1.04  -> abstain (no answer)   <- What is the diameter of the Moon?
span= 16.64  null=  4.81  -> answer                <- Who were the first humans to walk on the Moon?


## 3. Retrieve, then read — open-domain QA in miniature

Real questions don't come with a passage. **Open-domain QA** *retrieves* the relevant passage from a corpus, *then* reads a span out of it. Here is the two-stage pipeline in miniature: a transparent bag-of-words retriever (a stand-in for DPR's learned encoder) scores each passage by cosine to the question, and the **same span decode** reads the winner.

In [9]:
from question_answering import RETRIEVER_QUESTION, RETRIEVER_CORPUS, RETRIEVER_GOLD, retrieve_top_passage, read_span

top_idx, cosines = retrieve_top_passage(RETRIEVER_QUESTION, RETRIEVER_CORPUS)
answer = read_span(RETRIEVER_QUESTION, RETRIEVER_CORPUS[top_idx])

# only passage 0 actually answers the question — the retriever must surface it, the reader must find the name
assert top_idx == 0, f"retriever should pick passage 0, picked {top_idx}"
assert RETRIEVER_GOLD.lower() in answer.lower(), f"reader missed the answer in {answer!r}"

print(f"question: {RETRIEVER_QUESTION!r}\n")
for k, (passage, cos) in enumerate(zip(RETRIEVER_CORPUS, cosines)):
    mark = "  <-- retrieved (top cosine)" if k == top_idx else ""
    print(f"  cos={cos:.3f}  p{k}: {passage[:50]}...{mark}")
print(f"\nreader span -> {answer!r}")

question: 'Who painted the Mona Lisa?'

  cos=0.522  p0: The Mona Lisa is a half-length portrait painting b...  <-- retrieved (top cosine)
  cos=0.408  p1: The Louvre in Paris is the world's most-visited mu...
  cos=0.000  p2: Renaissance portraiture emphasized realistic depic...
  cos=0.000  p3: Photosynthesis converts sunlight, water, and carbo...

reader span -> 'Leonardo da Vinci'


### The retrieval recall ceiling

Open-domain accuracy is **bottlenecked by the retriever**: if the answer-bearing passage isn't in the top-$k$, a perfect reader still scores zero. End-to-end accuracy can never exceed retriever recall@k — the [`qa_retrieval_ceiling.png`](../images/qa_retrieval_ceiling.png) curve. We make the ceiling literal: drop the right passage out of reach and watch the system fail no matter how good the reader is.

In [10]:
# simulate a retriever that returns only top-1, and the right passage ranked 2nd -> a miss
ranked = sorted(range(len(RETRIEVER_CORPUS)), key=lambda k: -cosines[k])
k = 1
retrieved = set(ranked[:k])
answer_passage = 0  # passage 0 is the only one bearing the answer

hit = answer_passage in retrieved
print(f"top-{k} retrieved passages: {sorted(retrieved)}   answer is in passage {answer_passage}")
print(f"recall@{k} = {'HIT' if hit else 'MISS'}")
# with the real corpus passage 0 IS rank-1, so top-1 hits; force a miss to show the ceiling
forced_retrieved = {ranked[1]}   # pretend the retriever ranked the wrong passage first
forced_hit = answer_passage in forced_retrieved
assert not forced_hit, "forced miss: the answer passage is outside top-1"
print(f"\nif the retriever had ranked passage {ranked[1]} first instead: recall@1 = MISS")
print("-> a perfect reader scores 0 here: it never sees the evidence. The retriever is the ceiling.")

top-1 retrieved passages: [0]   answer is in passage 0
recall@1 = HIT

if the retriever had ranked passage 1 first instead: recall@1 = MISS
-> a perfect reader scores 0 here: it never sees the evidence. The retriever is the ceiling.


## 4. Live extractive QA on a real fine-tuned model

Finally, the real thing: the **derived decode** — `argmax_{j>=i}(start_i + end_j)` over the passage tokens — run against a real fine-tuned SQuAD model (`distilbert-base-cased-distilled-squad`) if it is cached, or the deterministic synthetic fallback otherwise. No library QA helper: this is our own decode on the model's raw logits, reproducing the page's measured table.

In [11]:
from question_answering import load_qa_model, answer_question, APOLLO_PASSAGE, APOLLO_QUESTIONS

backend = load_qa_model()
tag = f"real model ({backend.name})" if backend.is_real else "synthetic fallback (offline)"
print(f"backend: {tag}\n")
print(f"passage: {APOLLO_PASSAGE}\n")

for q in APOLLO_QUESTIONS:
    text, span_s, null_s = answer_question(backend, q, APOLLO_PASSAGE)
    # whatever the backend, the span score must beat the null score on these answerable questions
    assert span_s > null_s, f"answerable question should not abstain: {q}"
    print(f"  {text:34s}  span={span_s:6.2f}  null={null_s:6.2f}  <- {q}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

backend: real model (distilbert-base-cased-distilled-squad)

passage: The Apollo 11 mission landed the first humans on the Moon on July 20, 1969. Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited above.



  Neil Armstrong and Buzz Aldrin      span= 17.99  null= -7.65  <- Who were the first humans to walk on the Moon?
  July 20, 1969                       span= 23.74  null= -2.26  <- When did Apollo 11 land on the Moon?
  Michael Collins                     span= 20.05  null= -5.26  <- Who stayed in orbit during Apollo 11?


## Recap

Three machines, one lineage:

- **Span decode** — extractive QA is *pointing*: learn $\mathbf{S},\mathbf{E}$, score each token, and pick `argmax_{j>=i}(S·h_i + E·h_j)`. The `j >= i` constraint is load-bearing; abstaining is the `[CLS]` null span (SQuAD 2.0).
- **Scoring** — **EM** is all-or-nothing, **token F1** gives partial credit; both normalize first, and F1's leniency is exactly the slow precision decay we measured.
- **Retrieve-then-read** — open-domain QA bolts a retriever in front of the same reader; its accuracy is **capped by retrieval recall**, and the reader is the span head again, applied to whatever was retrieved.

Every number above is recomputed live from [`question_answering.py`](question_answering.py); the figures on the page come from the same functions via [`make_figures_11.py`](make_figures_11.py). For the full treatment — SQuAD 2.0, DPR, RAG, multi-hop, conversational QA, and the evaluation problem — see the **[Question Answering](../11-Question-Answering.md)** concept page.